# Geomagnetic Regime Examples Figure

Builds `fig:different_types_trajectories`: one real example sequence per evaluation regime
(high/low activity x high/low dynamism), picked from the actual metadata produced by the
paper's singlepass CLASSIC evaluation run, so the regimes and thresholds match the paper exactly
(not recomputed/approximated).

In [1]:

import sys, os, json, glob
sys.path.insert(0, '/users/framunno/projects/ionosphere_diffusion')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import map_coordinates
from src.data.dataset import latlon_to_cartesian_grid

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'STIXGeneral'

PAPER_FIGURES_DIR = '/users/framunno/projects/paper_writing/SuperDARN_deep_Learning_Francesco/figures'
CSV_PATH = '/users/framunno/data/ionosphere/l1_to_map_matched_2020_2025.csv'
LOCAL_MAPS = '/users/framunno/data/ionosphere/ionosphere_data/pickled_maps'

REGIMES = ['dynhigh_acthigh', 'dynhigh_actlow', 'dynlow_acthigh', 'dynlow_actlow']
REGIME_LABELS = {
    'dynhigh_acthigh': 'high activity / high dynamism',
    'dynhigh_actlow':  'low activity / high dynamism',
    'dynlow_acthigh':  'high activity / low dynamism',
    'dynlow_actlow':   'low activity / low dynamism',
}
META_BASE = '/capstor/scratch/cscs/framunno/results_diffusion_classic_singlepass_step200k_{regime}/metadata'


In [2]:

df = pd.read_csv(CSV_PATH, usecols=['filename', 'time_map', 'bx_gsm', 'by_gsm', 'bz_gsm', 'proton_vx_gsm'])
df['time_map'] = pd.to_datetime(df['time_map'])
df = df.dropna(subset=['time_map']).drop_duplicates(subset='filename').sort_values('time_map').reset_index(drop=True)
time_to_row = {t: i for i, t in enumerate(df['time_map'])}
print('csv rows:', len(df))


csv rows: 1370626


In [3]:

def load_candidates(regime):
    d = META_BASE.format(regime=regime)
    metas = []
    for f in glob.glob(os.path.join(d, '*.json')):
        m = json.load(open(f))
        st = pd.Timestamp(m['start_time'])
        if st.year == 2024:
            metas.append(m)
    metas.sort(key=lambda m: m['start_time'])
    return metas

for r in REGIMES:
    cands = load_candidates(r)
    print(r, '-> candidates in 2024:', len(cands))


dynhigh_acthigh -> candidates in 2024: 26


dynhigh_actlow -> candidates in 2024: 6


dynlow_acthigh -> candidates in 2024: 40


dynlow_actlow -> candidates in 2024: 17


In [4]:

def calculate_epsilon(vwind_kms, by, bz, l0=7e6):
    mu0 = 4 * np.pi * 1e-7
    vwind = np.abs(vwind_kms) * 1000
    by_T, bz_T = by * 1e-9, bz * 1e-9
    B_perp = np.sqrt(by_T**2 + bz_T**2)
    theta = np.arctan2(by_T, bz_T)
    eps = (vwind * B_perp**2 * l0**2 * np.sin(theta/2)**4) / mu0
    return eps / 1e9  # GW

def try_build_sequence(meta):
    start = pd.Timestamp(meta['start_time'])
    times = [start + pd.Timedelta(minutes=2*i) for i in range(22)]
    rows = []
    for t in times:
        if t not in time_to_row:
            return None
        rows.append(df.iloc[time_to_row[t]])
    frames = []
    for row in rows:
        fpath = os.path.join(LOCAL_MAPS, os.path.basename(row['filename']))
        if not os.path.exists(fpath):
            return None
        raw = np.load(fpath, allow_pickle=True)[0].astype(np.float32)
        frames.append(latlon_to_cartesian_grid(raw, output_size=128))
    frames = np.stack(frames)  # (22,128,128), physical Volts already (raw maps, not normalized)
    eps_vals = [calculate_epsilon(r['proton_vx_gsm'], r['by_gsm'], r['bz_gsm']) for r in rows]
    activity_score = float(np.max(eps_vals))
    dynamism_score = float(np.abs(np.diff(frames, axis=0)).mean())
    return {'frames': frames, 'times': times, 'activity': activity_score, 'dynamism': dynamism_score}

REGIME_DIRS = {
    'dynhigh_acthigh': dict(act='high', dyn='high'),
    'dynhigh_actlow':  dict(act='low',  dyn='high'),
    'dynlow_acthigh':  dict(act='high', dyn='low'),
    'dynlow_actlow':   dict(act='low',  dyn='low'),
}

chosen = {}
for r in REGIMES:
    valid = []
    for meta in load_candidates(r):
        res = try_build_sequence(meta)
        if res is not None:
            res['meta'] = meta
            valid.append(res)
    if not valid:
        print(r, 'NO VALID CANDIDATE FOUND WITH LOCAL FILES')
        continue

    acts = np.array([v['activity'] for v in valid])
    dyns = np.array([v['dynamism'] for v in valid])
    act_z = (acts - acts.mean()) / (acts.std() + 1e-9)
    dyn_z = (dyns - dyns.mean()) / (dyns.std() + 1e-9)
    if REGIME_DIRS[r]['act'] == 'low':
        act_z = -act_z
    if REGIME_DIRS[r]['dyn'] == 'low':
        dyn_z = -dyn_z
    extremity = act_z + dyn_z

    best_i = int(np.argmax(extremity))
    chosen[r] = valid[best_i]
    print(r, 'candidates:', len(valid), '-> picked start:', valid[best_i]['meta']['start_time'],
          'activity(GW):', round(valid[best_i]['activity'], 2),
          'dynamism(V/frame):', round(valid[best_i]['dynamism'], 1),
          '(most extreme of', len(valid), ')')


dynhigh_acthigh candidates: 26 -> picked start: 2024-10-10 19:00:00 activity(GW): 13.33 dynamism(V/frame): 6472.9 (most extreme of 26 )


dynhigh_actlow candidates: 6 -> picked start: 2024-10-28 01:42:00 activity(GW): 0.01 dynamism(V/frame): 3591.5 (most extreme of 6 )


dynlow_acthigh candidates: 40 -> picked start: 2024-10-11 09:24:00 activity(GW): 8.55 dynamism(V/frame): 811.1 (most extreme of 40 )


dynlow_actlow candidates: 17 -> picked start: 2024-10-14 00:14:00 activity(GW): 0.0 dynamism(V/frame): 319.3 (most extreme of 17 )


In [5]:

H_IMG, MAX_R = 128, 24
r_i, th_i = np.linspace(0, MAX_R, 200), np.linspace(0, 2*np.pi, 360)
r_grid, theta_grid = np.meshgrid(r_i, th_i)
polar_x, polar_y = r_grid*np.cos(theta_grid), r_grid*np.sin(theta_grid)
col_coords = (polar_x + MAX_R) / (2*MAX_R) * (H_IMG - 1)
row_coords = (polar_y + MAX_R) / (2*MAX_R) * (H_IMG - 1)

def to_polar(frame2d):
    return map_coordinates(frame2d, [row_coords, col_coords], order=1, mode='constant', cval=0)

VMAX = max(np.abs(chosen[r]['frames']).max() for r in REGIMES)
print('shared VMAX:', VMAX)


shared VMAX: 41985.70481623591


In [6]:

N_SNAPSHOTS = 7
snapshot_idx = np.linspace(0, 21, N_SNAPSHOTS).round().astype(int)

fig = plt.figure(figsize=(14.5, 11))
gs_outer = gridspec.GridSpec(4, 1, hspace=0.65, left=0.06, right=0.90, top=0.92, bottom=0.05, figure=fig)

last_mesh = None
for ri, r in enumerate(REGIMES):
    data = chosen[r]
    gs_row = gridspec.GridSpecFromSubplotSpec(1, N_SNAPSHOTS, subplot_spec=gs_outer[ri], wspace=0.12)
    for ci, fi in enumerate(snapshot_idx):
        ax = fig.add_subplot(gs_row[ci], projection='polar')
        last_mesh = ax.pcolormesh(theta_grid, r_grid, to_polar(data['frames'][fi]), shading='auto',
                                   cmap='coolwarm', vmin=-VMAX, vmax=VMAX)
        ax.set_theta_zero_location('S')
        ax.set_theta_direction(1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.spines['polar'].set_linewidth(0.8)
        ax.spines['polar'].set_color('0.55')

    fig.canvas.draw()
    row_axes = fig.axes[-N_SNAPSHOTS:]
    x_mid = (row_axes[0].get_position().x0 + row_axes[-1].get_position().x1) / 2
    y_top = row_axes[0].get_position().y1 + 0.02
    fig.text(x_mid, y_top, REGIME_LABELS[r], fontsize=16, fontweight='bold', ha='center', va='bottom',
              fontfamily='STIXGeneral')

cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(last_mesh, cax=cbar_ax)
cbar.set_label('Electric potential [V]', fontsize=18, fontfamily='STIXGeneral')
cbar.ax.tick_params(labelsize=15)

from matplotlib.patches import FancyArrowPatch

bottom_row_axes = fig.axes[-N_SNAPSHOTS-1:-1]  # last regime row (colorbar axis is fig.axes[-1])
x0 = bottom_row_axes[0].get_position().x0
x1 = bottom_row_axes[-1].get_position().x1
y_arrow = bottom_row_axes[0].get_position().y0 - 0.045
arrow = FancyArrowPatch((x0, y_arrow), (x1, y_arrow), transform=fig.transFigure,
                         arrowstyle='-|>', mutation_scale=16, color='0.25', lw=1.4)
fig.add_artist(arrow)
fig.text((x0 + x1) / 2, y_arrow - 0.025, 'Time', fontsize=14, ha='center', va='top',
          color='0.25', fontfamily='STIXGeneral')

plt.savefig(f'{PAPER_FIGURES_DIR}/methodology_regime_examples.png', dpi=300, bbox_inches='tight')
plt.show()


/tmp/ipykernel_191561/1170972408.py:13: MatplotlibDeprecationWarning: Auto-removal of grids by pcolor() and pcolormesh() is deprecated since 3.5 and will be removed two minor releases later; please call grid(False) first.
  last_mesh = ax.pcolormesh(theta_grid, r_grid, to_polar(data['frames'][fi]), shading='auto',
